# PPO：最流行的策略梯度算法
> 难度标签：高级 | 预计时长：20-30分钟 | 前置知识：无学习经验


> 所属模块：05_强化学习 | 源文件：PPO.py | 核心功能：裁剪机制、GAE 优势估计、多 epoch 更新、OpenAI 标准实现

## 概述

PPO（Proximal Policy Optimization）是 OpenAI 2017 年提出的策略梯度算法，目前是强化学习领域**使用最广泛**的算法。它的成功来自两个特点：训练稳定性好 + 实现简单。

核心创新是**裁剪机制**：用概率比 r(theta) = pi_new/pi_old 衡量策略变化幅度，然后裁剪到 [1-epsilon, 1+epsilon] 范围内，防止策略更新过大导致崩溃。

脚本实现了完整的 PPO 训练流程：经验收集、GAE 优势估计、裁剪损失、多 epoch 更新。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

## 数学原理

### 1. 重要性采样与策略比

**代码对应**：PPO 通过限制策略更新幅度实现稳定训练。

新策略 $\pi_\theta$ 与旧策略 $\pi_{\theta_{\text{old}}}$ 之间的**策略比**：

$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

用重要性采样，旧策略采集的数据可用于更新新策略：

$$J(\theta) = \mathbb{E}\left[r_t(\theta) \cdot A_t\right]$$

### 2. PPO-Clip 目标函数

**代码对应**：PPO 的核心是裁剪目标函数，限制策略变化幅度。

$$L^{\text{CLIP}}(\theta) = \mathbb{E}\left[\min\left(r_t(\theta)A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)A_t\right)\right]$$

其中 $\epsilon$ 为裁剪参数（通常 0.1~0.2）。

**直觉**：
- 当 $A_t > 0$（好动作）：$r_t(\theta)$ 被裁剪到 $[1-\epsilon, 1+\epsilon]$，防止策略变化过大
- 当 $A_t < 0$（坏动作）：同理，限制策略远离该动作的幅度

### 3. 广义优势估计（GAE）

$$A_t^{\text{GAE}(\gamma, \lambda)} = \sum_{l=0}^{\infty}(\gamma\lambda)^l\delta_{t+l}$$

其中 $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ 为 TD 误差。

$\lambda \in [0, 1]$ 控制偏差-方差权衡：$\lambda = 0$ 为 TD(0)（低方差高偏差），$\lambda = 1$ 为蒙特卡洛（高方差低偏差）。

### 4. PPO 的完整损失

$$L(\theta) = L^{\text{CLIP}}(\theta) + c_1 L^{\text{VF}}(\theta) - c_2 H(\pi_\theta)$$

- $L^{\text{CLIP}}$：策略损失（裁剪目标）
- $L^{\text{VF}}$：值函数损失 $(V_\theta(s) - V_{\text{target}})^2$
- $H(\pi_\theta)$：熵正则化（鼓励探索）

### 5. 为什么 PPO 如此流行

PPO 是 OpenAI 的默认强化学习算法，因为：
- 实现简单（比 TRPO 不需要二阶优化）
- 训练稳定（裁剪限制策略变化）
- 样本效率较高（可多次复用同一批数据）
- 适用范围广（连续/离散动作空间均可）

### 1. 简单 CartPole 环境

运行 1. 简单 CartPole 环境 的代码，观察算法在该环节的行为。

In [ ]:
class SimpleCartPole:
    def __init__(self):
        self.max_steps = 200
        self.reset()

    def reset(self):
        self.state = np.random.uniform(-0.05, 0.05, 4)
        self.steps = 0
        return self.state.copy()

    def step(self, action):
        x, x_dot, theta, theta_dot = self.state
        force = 1.0 if action == 1 else -1.0
        x_ddot = (force - 0.0025 * x_dot + 0.001 * np.sin(theta)) / 1.0
        theta_ddot = (0.01 * np.sin(theta) - 0.001 * theta_dot + 0.0001 * force) / 0.1
# --- 赋值/配置 ---
        dt = 0.02
        x_dot += x_ddot * dt
        theta_dot += theta_ddot * dt
        x += x_dot * dt
        theta += theta_dot * dt
# --- 数值计算 ---
        self.state = np.array([x, x_dot, theta, theta_dot])
        self.steps += 1
        done = abs(x) > 2.4 or abs(theta) > 0.21 or self.steps >= self.max_steps
        reward = 1.0 if not done else 0.0
        return self.state.copy(), reward, done

### 2. PPO 网络

运行 2. PPO 网络 的代码，观察算法在该环节的行为。

In [ ]:
class PPOActorCritic(nn.Module):
    def __init__(self, state_dim=4, action_dim=2, hidden=64):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden),
# --- 继续 ---
            nn.ReLU(),
        )
        self.actor = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
# --- 继续 ---
            nn.Linear(hidden, action_dim),
            nn.Softmax(dim=-1),
        )
        self.critic = nn.Sequential(
            nn.Linear(hidden, hidden),
# --- 继续 ---
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        features = self.shared(x)
        return self.actor(features), self.critic(features)

    def get_action(self, state):
        probs, value = self.forward(state)
        dist = Categorical(probs)
        action = dist.sample()
        return action, dist.log_prob(action), value

### 3. PPO 算法

运行 3. PPO 算法 的代码，观察算法在该环节的行为。

In [ ]:
class PPOAgent:
    def __init__(self, state_dim=4, action_dim=2, lr=3e-4, gamma=0.99,
                 clip_epsilon=0.2, epochs=10, batch_size=64):
        self.gamma = gamma
        self.clip_epsilon = clip_epsilon
# --- 赋值/配置 ---
        self.epochs = epochs
        self.batch_size = batch_size
        self.model = PPOActorCritic(state_dim, action_dim)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

    def compute_gae(self, rewards, values, dones, next_value):
        """广义优势估计（GAE）"""
        advantages = []
        gae = 0
        values = values + [next_value]
# --- 循环处理 ---
        for t in reversed(range(len(rewards))):
            delta = rewards[t] + self.gamma * values[t + 1] * (1 - dones[t]) - values[t]
            gae = delta + self.gamma * 0.95 * (1 - dones[t]) * gae  # lambda=0.95
            advantages.insert(0, gae)
        return advantages

    def update(self, states, actions, old_log_probs, returns, advantages):
        """PPO 核心更新"""
        states = torch.FloatTensor(np.array(states))
        actions = torch.LongTensor(actions)
        old_log_probs = torch.FloatTensor(old_log_probs)
# --- 继续 ---
        returns = torch.FloatTensor(returns)
        advantages = torch.FloatTensor(advantages)
        # 标准化优势
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        total_loss = 0
        for _ in range(self.epochs):
            # 随机打乱
            indices = np.random.permutation(len(states))
            for start in range(0, len(states), self.batch_size):
                end = start + self.batch_size
                batch_idx = indices[start:end]

                probs, values = self.model(states[batch_idx])
                dist = Categorical(probs)
                new_log_probs = dist.log_prob(actions[batch_idx])

                # PPO 裁剪目标
                ratio = torch.exp(new_log_probs - old_log_probs[batch_idx])
                clipped_ratio = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon)

                # Actor 损失（取较小值，保守更新）
                surr1 = ratio * advantages[batch_idx]
                surr2 = clipped_ratio * advantages[batch_idx]
                actor_loss = -torch.min(surr1, surr2).mean()

                # Critic 损失
                critic_loss = nn.MSELoss()(values.squeeze(), returns[batch_idx])

                loss = actor_loss + 0.5 * critic_loss
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 0.5)
                self.optimizer.step()
# --- 继续 ---
                total_loss += loss.item()

        return total_loss

### 4. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
env = SimpleCartPole()
agent = PPOAgent()
update_every = 128  # 每收集 128 步更新一次

print("=== PPO 训练 ===")
n_episodes = 500
reward_history = []
total_steps = 0

# 收集经验
states_buf, actions_buf, rewards_buf, log_probs_buf, values_buf, dones_buf = [], [], [], [], [], []

for episode in range(n_episodes):
    state = env.reset()
    episode_reward = 0
    done = False

    while not done:
        state_t = torch.FloatTensor(state).unsqueeze(0)
        action, log_prob, value = agent.model.get_action(state_t)
        action = action.item()
        next_state, reward, done = env.step(action)

        states_buf.append(state)
        actions_buf.append(action)
        rewards_buf.append(reward)
        log_probs_buf.append(log_prob.item())
        values_buf.append(value.item())
# --- 继续 ---
        dones_buf.append(done)

        state = next_state
        episode_reward += reward
        total_steps += 1

        # 定期更新
        if total_steps % update_every == 0 and len(states_buf) >= update_every:
            with torch.no_grad():
                next_state_t = torch.FloatTensor(next_state).unsqueeze(0)
                _, next_value = agent.model(next_state_t)
                next_value = next_value.item()

            advantages = agent.compute_gae(rewards_buf, values_buf, dones_buf, next_value)
            returns = [a + v for a, v in zip(advantages, values_buf)]

            agent.update(states_buf, actions_buf, log_probs_buf, returns, advantages)

            states_buf, actions_buf, rewards_buf = [], [], []
            log_probs_buf, values_buf, dones_buf = [], [], []

    reward_history.append(episode_reward)
    if (episode + 1) % 50 == 0:
        avg = np.mean(reward_history[-50:])
        print(f"  Episode {episode+1:>3}: 平均奖励={avg:>6.1f}, 总步数={total_steps}")

### 5. 测试

运行 5. 测试 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 测试 ===")
test_rewards = []
for _ in range(10):
    state = env.reset()
    total_reward = 0
# --- 赋值/配置 ---
    done = False
    while not done:
        state_t = torch.FloatTensor(state).unsqueeze(0)
        probs, _ = agent.model(state_t)
        action = probs.argmax().item()
# --- 继续 ---
        state, reward, done = env.step(action)
        total_reward += reward
    test_rewards.append(total_reward)
print(f"10 次测试平均奖励: {np.mean(test_rewards):.1f}")

### 6. PPO 核心机制

运行 6. PPO 核心机制 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== PPO 核心机制 ===")
print("裁剪目标: L = min(r(θ)×A, clip(r(θ), 1-ε, 1+ε)×A)")
print("  r(θ) = π_new(a|s) / π_old(a|s)  — 新旧策略的概率比")
print("  A = 优势函数（GAE 计算）")
print("  ε = 裁剪范围（通常 0.2）")
# --- 输出结果 ---
print()
print("裁剪的作用:")
print("- 当 A>0（好动作）: r 被裁剪到 1+ε，防止过度增加概率")
print("- 当 A<0（坏动作）: r 被裁剪到 1-ε，防止过度降低概率")
print("- 效果：策略更新幅度被限制，训练更稳定")

print("\n=== PPO 要点 ===")
print("- 结合了 TRPO 的稳定性和实现的简单性")
print("- clip_epsilon: 核心超参数，通常 0.1~0.3")
print("- epochs: 每批数据可复用多次，提高样本效率")
print("- GAE: 平衡偏差和方差的优势估计方法")
# --- 输出结果 ---
print("- 目前最流行的策略梯度算法之一（适用于连续和离散动作空间）")

## 关键代码解释

### 裁剪目标

```python
ratio = torch.exp(new_log_probs - old_log_probs)
clipped_ratio = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon)
actor_loss = -torch.min(ratio * advantages, clipped_ratio * advantages).mean()
```

取 min 意味着：当策略变化方向正确时（A>0），最多让概率增加 epsilon 倍；方向错误时（A<0），最多让概率减少 epsilon 倍。

### GAE 优势估计

```python
delta = reward + gamma * V(next_state) - V(state)
gae = delta + gamma * lambda * gae
```

GAE 通过 lambda 参数平衡偏差和方差。lambda=0 退化为 TD，lambda=1 退化为蒙特卡洛。

## 使用示例

```python
agent = PPOAgent()
# 收集经验
for step in range(update_every):
    action, log_prob, value = agent.model.get_action(state)
# 更新
agent.update(states, actions, old_log_probs, returns, advantages)
```

## 注意事项

1. **clip_epsilon 通常取 0.1-0.3**
2. **多 epoch 复用数据**：同一批数据更新多次，提高样本效率
3. **GAE lambda 通常取 0.95**
4. **梯度裁剪**：clip_grad_norm 防止梯度爆炸

## 总结与延伸

以上代码展示了 **PPO** 的完整流程。

**解读要点**：
- 关注 **累计奖励** 随训练轮次的增长趋势
- 观察 **探索率 epsilon** 的衰减过程
- 对比不同策略（greedy vs epsilon-greedy）的表现

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **PPO for LLM**：ChatGPT 的 RLHF 阶段就使用了 PPO
- **PPO-Clip vs PPO-Penalty**：两种 PPO 变体
- **分布式 PPO**：如 IMPALA、Distributed PPO
- **Mamba 等替代方案**：状态空间模型能否替代 Transformer+PPO
